In [ ]:
!pip install -q yfinance tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

TICKER       = "AAPL"
START        = "2018-01-01"
END          = "2024-12-31"
TARGET       = "Close"
TIME_STEPS   = 30
VALID_RATIO  = 0.2
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def load_stock_data(ticker=TICKER, start=START, end=END):
    # 1. yfinance
    try:
        raw = yf.download(ticker, start=start, end=end,
                          progress=False, auto_adjust=True)
        if not raw.empty:
            if isinstance(raw.columns, pd.MultiIndex):
                lvl0 = raw.columns.get_level_values(0).unique().tolist()
                raw.columns = (raw.columns.get_level_values(0)
                               if "Close" in lvl0
                               else raw.columns.get_level_values(-1))
            raw = raw.reset_index()
            if "Close" in raw.columns and len(raw) > 100:
                print(f"✓ yfinance OK — {len(raw)} dòng")
                return raw
    except Exception as e:
        print(f"✘ yfinance: {e}")

    # 2. stooq
    print("→ thử stooq.com...")
    try:
        url = f"https://stooq.com/q/d/l/?s={ticker.lower()}.us&i=d"
        raw = pd.read_csv(url)
        raw["Date"] = pd.to_datetime(raw["Date"])
        raw = raw[(raw["Date"] >= start) & (raw["Date"] <= end)].reset_index(drop=True)
        if "Close" in raw.columns and len(raw) > 100:
            print(f"✓ stooq OK — {len(raw)} dòng")
            return raw
    except Exception as e:
        print(f"✘ stooq: {e}")

    # 3. tổng hợp (offline)
    print("→ dữ liệu tổng hợp (offline)")
    dates = pd.date_range(start, end, freq="B")
    n = len(dates)
    close = np.linspace(100, 250, n) + 10*np.sin(np.arange(n)*2*np.pi/252) \
            + np.random.normal(0, 3, n)
    return pd.DataFrame({
        "Date": dates,
        "Open":   close + np.random.normal(0, 1, n),
        "High":   close + np.abs(np.random.normal(2, 1, n)),
        "Low":    close - np.abs(np.random.normal(2, 1, n)),
        "Close":  close,
        "Volume": np.random.randint(1_000_000, 50_000_000, n),
    })

df = load_stock_data()
display(df.head())
print("Số dòng:", len(df))

ERROR:yfinance:Failed to get ticker 'AAPL' reason: Impersonating chrome150 is not supported
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: YFTzMissingError('possibly delisted; no timezone found')


→ thử stooq.com...
✘ stooq: HTTP Error 404: Not Found
→ dữ liệu tổng hợp (offline)


,Date,Open,High,Low,Close,Volume
0,2018-01-01,102.901075,103.639506,99.039529,101.490142,22783250
1,2018-01-02,99.818073,102.280801,97.833888,99.916661,18454767
2,2018-01-03,102.624668,104.612996,100.569035,102.605818,18989956
3,2018-01-04,106.271045,105.649308,105.178524,105.562831,24158079
4,2018-01-05,100.855022,102.782295,97.876655,100.621805,25323577


Số dòng: 1827


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"]      = df["Date"].dt.year
df["Month"]     = df["Date"].dt.month
df["Day"]       = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["sin_dow"]   = np.sin(2*np.pi*df["DayOfWeek"]/7)
df["cos_dow"]   = np.cos(2*np.pi*df["DayOfWeek"]/7)
df["sin_month"] = np.sin(2*np.pi*df["Month"]/12)
df["cos_month"] = np.cos(2*np.pi*df["Month"]/12)

for lag in [1, 2, 3, 5, 7]:
    df[f"Close_lag{lag}"] = df["Close"].shift(lag)

before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"dropna: {before} → {len(df)} dòng")
assert len(df) > 100, "Dữ liệu quá ít — kiểm tra CELL 3."

dropna: 1827 → 1820 dòng


In [ ]:
FEATURES = [
    "Open", "High", "Low", "Volume",
    "Year", "Month", "Day", "DayOfWeek",
    "sin_dow", "cos_dow", "sin_month", "cos_month",
    "Close_lag1", "Close_lag2", "Close_lag3", "Close_lag5", "Close_lag7",
]

split_idx = int(len(df) * (1 - VALID_RATIO))
train_df = df.iloc[:split_idx].copy()
valid_df = df.iloc[split_idx:].copy()

print(f"Train: {train_df['Date'].min().date()} → {train_df['Date'].max().date()} ({len(train_df):,})")
print(f"Valid: {valid_df['Date'].min().date()} → {valid_df['Date'].max().date()} ({len(valid_df):,})")

Train: 2018-01-10 → 2023-08-09 (1,456)
Valid: 2023-08-10 → 2024-12-31 (364)


In [ ]:
X_train_lr = train_df[FEATURES].fillna(0).values
y_train_lr = train_df[TARGET].values
X_valid_lr = valid_df[FEATURES].fillna(0).values
y_valid_lr = valid_df[TARGET].values

lr = LinearRegression().fit(X_train_lr, y_train_lr)
pred_lr = lr.predict(X_valid_lr)

def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2":   r2_score(y_true, y_pred),
    }

results = [evaluate("LinearRegression", y_valid_lr, pred_lr)]
display(pd.DataFrame(results).round(3))

,Model,MAE,RMSE,R2
0,LinearRegression,0.468,0.578,0.997


In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaled = df[FEATURES + [TARGET]].copy()
scaled[FEATURES]   = scaler_X.fit_transform(scaled[FEATURES])
scaled[[TARGET]]   = scaler_y.fit_transform(scaled[[TARGET]])

def create_sequences(data, features, target, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[features].iloc[i:i+time_steps].values)
        y.append(data[target].iloc[i+time_steps])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(scaled, FEATURES, TARGET, TIME_STEPS)
n_valid = int(len(X_all) * VALID_RATIO)

X_train_lstm, y_train_lstm = X_all[:-n_valid], y_all[:-n_valid]
X_valid_lstm, y_valid_lstm = X_all[-n_valid:], y_all[-n_valid:]

print("LSTM Train:", X_train_lstm.shape)
print("LSTM Valid:", X_valid_lstm.shape)

LSTM Train: (1432, 30, 17)
LSTM Valid: (358, 30, 17)


In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(TIME_STEPS, len(FEATURES))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dense(1),
])
model.compile(optimizer="adam", loss="mse")
model.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=8,
                           restore_best_weights=True)

history = model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_valid_lstm, y_valid_lstm),
    epochs=30, batch_size=64,
    callbacks=[early_stop], verbose=1,
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,441 (130.63 KB)

 Trainable params: 33,441 (130.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 120ms/step - loss: 0.0208 - val_loss: 0.0023
Epoch 2/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.0021 - val_loss: 0.0054
Epoch 3/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0012 - val_loss: 0.0034
Epoch 4/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 9.4871e-04 - val_loss: 0.0025
Epoch 5/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 9.2631e-04 - val_loss: 0.0013
Epoch 6/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 8.7586e-04 - val_loss: 0.0022
Epoch 7/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 8.2524e-04 - val_loss: 0.0013
Epoch 8/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 7.3482e-04 - val_loss: 0.0020
Epoch 9/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 7.6161e-04 - val_loss: 0.0017
Epoch 10/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 6.7411e-04 - val_loss: 9.4053e-04
Epoch 11/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 6.6677e-04 - val_loss: 0.0011
Epoch 12/30
23/23 ━━━━━━━━